In [2]:
!pip install spacy pandas pdfplumber python
!python -m spacy download en_core_web_trf

Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement python (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: C:\Users\irind\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for python


Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.8/457.4 MB 6.7 MB/s eta 0:01:08
     ---------------------------------------- 2.4/457.4 MB 6.4 MB/s eta 0:01:12
     ---------------------------------------- 4.7/457.4 MB 8.2 MB/s eta 0:00:56
      --------------------------------------- 5.8/457.4 MB 7.2 MB/s eta 0:01:03
      --------------------------------------- 7.6/457.4 MB 8.0 MB/s eta 0:00:57
      --------------------------------------- 9.2/457.4 MB 7.8 MB/s eta 0:00:58
      -------------------------------------- 11.0/457.4 MB 7.7 MB/s eta 0:00:58
     - ------------------------------------- 12.3/457.4 MB 7.6 MB/s eta 0:00:59
     - ------------------------------------- 13.4/457.4 MB 7.4 MB/s eta 0:01:01
     - ------------------------------------- 14.2/457.4 MB 7.1 MB/s eta 0:01:03
     - ------------------------------------- 15.2


[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: C:\Users\irind\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import os
import re
import spacy 
import pdfplumber
import pandas as pd


In [4]:
nlp = spacy.load("en_core_web_trf")

C:\Users\irind\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\irind\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\thinc\shims\pytorch.py:261: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during un

In [5]:
def extract_text_from_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf :
        text = "\n".join ([page.extract_text() for page in pdf.pages if page.extract_text()])
    return text 

In [6]:
def detect_full_caps_name(text):
    lines = text.strip().split("\n")[:7] 
    first_line = lines[0] if lines else ""
    print("\n Checking full caps & partial capitalozation")
    print(f" first line: {first_line}")

    if first_line.isupper() and 1 <= len(first_line.split()) <=4:
        print(f"\n Detected full caps name :{first_line.title()}\n")
        return first_line.title()
    
    if 1 <=len(first_line.split()) <=4:
        cap_words = [word for word in first_line.split() if word[0].isupper()]
        if len(cap_words) >=2:
            extracted_name = " ".join(cap_words)
            print(f"\n Detected name: {extracted_name}\n")
            return extracted_name
        
    print("\n No valid name detected\n")
    return None

In [7]:
def extract_name_from_labeled_section(text):
    """Extracts names from resumes where 'Name:' is explicitly mentioned."""
    match = re.search(r"(?i)Name[:\- ]+([A-Z][a-z]+(?: [A-Z][a-z]+)*)", text[:200])  # First 200 chars
    if match:
        return match.group(1)
    return None


In [8]:
def extract_name_spacy(text):
    first_lines = text.strip().split("\n")[:7]
    doc = nlp("\n".join(first_lines))

    for ent in doc.ents:
        if ent.label_ == "PERSON":
            return ent.text.title()
    return None


In [9]:
def extract_name_from_filename(filename):
    
    filename = filename.replace("_", " ").replace("-", " ").replace(".pdf", "").replace(".docx", "")
    words = filename.split()
    return " ".join(words[:2])  # Return first two words as name


In [10]:
def extract_name_final(text,filename):
    print(f"\n Extracting Name from :{filename}\n")

    name = detect_full_caps_name(text)
    if name:
        return name
    
    name = extract_name_from_labeled_section(text)
    if name:
        return name
    
    name = extract_name_spacy(text)
    if name:
        return name
    
    name = extract_name_from_filename(filename)

    return name
    

In [11]:
def extract_email(text):
    """Extracts email addresses from the text."""
    match = re.findall(r"[a-zA-Z0-9+_.-]+@[a-zA-Z0-9.-]+", text)
    return match[0] if match else None

def extract_phone(text):
    """Extracts phone numbers from the text."""
    match = re.findall(r"\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4,}", text)
    return match[0] if match else None


In [14]:
DEGREES = ["B.Sc", "BSc", "Bachelor of Science", "M.Sc", "MSc", "Master of Science",
           "PhD", "Doctorate", "MBA", "B.Tech", "M.Tech", "B.E", "M.E", "Diploma"]

def extract_education(text):
    """Extracts education details by searching for degree names."""
    found_degrees = [degree for degree in DEGREES if degree in text]
    return found_degrees if found_degrees else None


In [15]:
TECH_SKILLS = ["Python", "Machine Learning", "Deep Learning", "SQL", "Data Science",
               "TensorFlow", "PyTorch", "NLP", "Excel", "Power BI", "Tableau", "R",
               "Big Data", "Cloud Computing", "AWS", "Hadoop", "Spark", "Computer Vision"]

def extract_skills(text):
    """Extracts relevant skills from the resume."""
    found_skills = [skill for skill in TECH_SKILLS if skill.lower() in text.lower()]
    return found_skills if found_skills else None


In [16]:
EXPERIENCE_KEYWORDS = ["Experience", "Worked at", "Internship", "Employment", "Job", "Company"]

def extract_experience(text):
    """Extracts experience information."""
    experience_lines = [line for line in text.split("\n") if any(word in line for word in EXPERIENCE_KEYWORDS)]
    return experience_lines if experience_lines else None


In [18]:
def extract_resume_features(text, filename):
    """Extracts all features from a resume."""
    name = detect_full_caps_name(text) or \
           extract_name_from_labeled_section(text) or \
           extract_name_spacy(text) or \
           extract_name_from_filename(filename)

    email = extract_email(text)
    phone = extract_phone(text)
    education = extract_education(text)
    skills = extract_skills(text)
    experience = extract_experience(text)

    return {
        "Filename": filename,
        "Name": name,
        "Email": email,
        "Phone": phone,
        "Education": education,
        "Skills": skills,
        "Experience": experience
    }


In [21]:
import os
import pandas as pd

resume_folder = r"C:\Users\irind\OneDrive\Desktop\Resume_Parsing\Resume_Dataset"
resume_data = []

for filename in os.listdir(resume_folder):
    if filename.endswith(".pdf"):
        filepath = os.path.join(resume_folder, filename)
        text = extract_text_from_pdf(filepath)

        resume_info = extract_resume_features(text, filename)
        resume_data.append(resume_info)

df = pd.DataFrame(resume_data)
print(df.head())  # Display first few rows



 Checking full caps & partial capitalozation
 first line: ABHIRAM S

 Detected full caps name :Abhiram S


 Checking full caps & partial capitalozation
 first line: Abhirami A S

 Detected name: Abhirami A S


 Checking full caps & partial capitalozation
 first line: ADHITHYA M SURESH

 Detected full caps name :Adhithya M Suresh


 Checking full caps & partial capitalozation
 first line: ADITHYA A K

 Detected full caps name :Adithya A K


 Checking full caps & partial capitalozation
 first line: ADWAITH BIJU

 Detected full caps name :Adwaith Biju


 Checking full caps & partial capitalozation
 first line: Aflaha H

 Detected name: Aflaha H


 Checking full caps & partial capitalozation
 first line: AFSAL RAHMAN K

 Detected full caps name :Afsal Rahman K


 Checking full caps & partial capitalozation
 first line: AISWARYA R B

 Detected full caps name :Aiswarya R B


 Checking full caps & partial capitalozation
 first line: AKHUL MURALI

 Detected full caps name :Akhul Murali


 Che

In [22]:
df.to_csv("Extracted_Resume_Data.csv", index=False)
print("✅ Data saved successfully!")


✅ Data saved successfully!


In [24]:
def calculate_resume_score(resume, job_skills):
    """Scores resumes based on job match."""
    skills = resume["Skills"] if resume["Skills"] else []  # Handle NoneType
    experience = resume["Experience"] if resume["Experience"] else []
    education = resume["Education"] if resume["Education"] else []

    skill_match = len(set(skills) & set(job_skills))  # Count matching skills
    experience_score = len(experience)  # Number of experience entries
    education_score = len(education)  # Number of education degrees

    total_score = (skill_match * 3) + (experience_score * 2) + (education_score * 1)  # Adjust weights
    return total_score


In [25]:
job_skills = ["Python", "Machine Learning", "SQL", "Deep Learning"]

df["Resume Score"] = df.apply(lambda x: calculate_resume_score(x, job_skills), axis=1)

# Sort resumes by score
df = df.sort_values(by="Resume Score", ascending=False)
df.to_csv("Ranked_Resumes.csv", index=False)

print("✅ Resumes ranked & saved!")


✅ Resumes ranked & saved!


In [27]:
df["Skills"] = df["Skills"].apply(lambda x: x if isinstance(x, list) else [])  # Convert None to []


In [29]:
print(df.columns)  # List all available columns


Index(['Filename', 'Name', 'Email', 'Phone', 'Education', 'Skills',
       'Experience', 'Resume Score'],
      dtype='object')


In [31]:
job_roles = ["Data Scientist", "Data Analyst", "Software Engineer", "ML Engineer", "AI Researcher"]

# Repeat job roles to match dataframe length
df["Job Role"] = job_roles * (len(df) // len(job_roles)) + job_roles[:len(df) % len(job_roles)]


In [33]:
print(df["Job Role"].value_counts())  # Check job role distribution
print(df[["Skills", "Job Role"]].head(10))  # See job role vs. skills


Job Role
Data Scientist       16
Data Analyst         16
Software Engineer    16
ML Engineer          16
AI Researcher        15
Name: count, dtype: int64
                                               Skills           Job Role
13  [Python, Machine Learning, Deep Learning, SQL,...     Data Scientist
19  [Python, Machine Learning, Deep Learning, SQL,...       Data Analyst
47  [Python, Machine Learning, Deep Learning, SQL,...  Software Engineer
53  [Python, Machine Learning, Deep Learning, SQL,...        ML Engineer
27  [Python, Machine Learning, Deep Learning, SQL,...      AI Researcher
15  [Python, Machine Learning, Deep Learning, SQL,...     Data Scientist
54  [Python, Machine Learning, Deep Learning, Data...       Data Analyst
48  [Python, Machine Learning, Deep Learning, SQL,...  Software Engineer
10  [Python, Machine Learning, Deep Learning, SQL,...        ML Engineer
68  [Python, Machine Learning, Deep Learning, SQL,...      AI Researcher


In [34]:
def infer_job_role(row):
    """Predicts job role based on skills & education."""
    skills = set(row["Skills"]) if row["Skills"] else set()
    education = set(row["Education"]) if row["Education"] else set()

    if "Machine Learning" in skills or "Deep Learning" in skills:
        return "Data Scientist"
    if "SQL" in skills and ("Power BI" in skills or "Tableau" in skills):
        return "Data Analyst"
    if "Python" in skills and "TensorFlow" in skills:
        return "ML Engineer"
    if "Cloud Computing" in skills or "AWS" in skills:
        return "Software Engineer"
    if "PhD" in education:
        return "AI Researcher"
    
    return "Unknown"

df["Job Role"] = df.apply(infer_job_role, axis=1)


In [35]:
df["Education_Level"] = df["Education"].apply(lambda x: len(x) if isinstance(x, list) else 0)
df["Experience_Length"] = df["Experience"].apply(lambda x: len(x) if isinstance(x, list) else 0)


In [32]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

mlb = MultiLabelBinarizer()
X = mlb.fit_transform(df["Skills"])  # Convert skills into binary format
y = df["Job Role"]  # Target variable (e.g., "Data Scientist", "Software Engineer")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Evaluate accuracy
accuracy = model.score(X_test, y_test) * 100
print(f"✅ Model Accuracy: {accuracy:.2f}%")


✅ Model Accuracy: 12.50%
